In [164]:
# %load_ext autoreload
# %autoreload 2

In [46]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
books_features = ['book_id', 'text_reviews_count', 'series', 'popular_shelves', 'is_ebook', 'average_rating',
'similar_books', 'description', 'format', 'link', 'authors','publisher',
'num_pages', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'title',
'title_without_series']

In [10]:
books_df = pd.read_csv("../data/raw/goodreads_books_ENG.csv",
                       usecols=books_features)

books_df.head(2)

,text_reviews_count,series,popular_shelves,is_ebook,average_rating,similar_books,description,format,link,authors,publisher,num_pages,publication_year,url,image_url,book_id,ratings_count,title,title_without_series
0,7.0,['189911'],"[{'count': '58', 'name': 'to-read'}, {'count':...",False,4.03,"['19997', '828466', '1569323', '425389', '1176...",Omnibus book club edition containing the Ladie...,Hardcover,https://www.goodreads.com/book/show/7327624-th...,"[{'author_id': '10333', 'role': ''}]","Nelson Doubleday, Inc.",600.0,1987.0,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140.0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,3282.0,[],"[{'count': '7615', 'name': 'to-read'}, {'count...",False,3.49,"['6604176', '6054190', '2285777', '82641', '75...",Addie Downs and Valerie Adler were eight when ...,Hardcover,https://www.goodreads.com/book/show/6066819-be...,"[{'author_id': '9212', 'role': ''}]",Atria Books,368.0,2009.0,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...,6066819,51184.0,Best Friends Forever,Best Friends Forever


In [ ]:
import ast

books_df['total_shelves_count'] = (
    books_df['popular_shelves']
    .apply(lambda x: sum(int(item['count']) for item in ast.literal_eval(x)))
)
books_df['is_series'] = books_df['series'].apply(
    lambda x: 'Y' if len(ast.literal_eval(x)) > 0 else 'N'
)

In [9]:
books_df.shape

(866183, 19)

## Parallelize

In [12]:
from multiprocessing import  Pool
from functools import partial
import numpy as np
import pandas as pd


def parallelize(data, func, num_of_processes=8):
    data_split = np.array_split(data, num_of_processes)
    pool = Pool(num_of_processes)
    data = pd.concat(pool.map(func, data_split))
    pool.close()
    pool.join()
    return data

def run_on_subset(func, data_subset):
    return data_subset.apply(func, axis=1)

def parallelize_on_rows(data, func, num_of_processes=8):
    return parallelize(data, partial(run_on_subset, func), num_of_processes)

So the following line:

```python
df.apply(some_func, axis=1)
```

Will become:

```python
parallelize_on_rows(df, some_func)
``` 

In [13]:
def total_shelves_count(row):
    return sum(int(item['count']) for item in ast.literal_eval(row['popular_shelves']))

In [14]:
def is_series(row):
    return 'Y' if len(ast.literal_eval(row['series'])) > 0 else 'N'

In [17]:
books_df['total_shelves_count'] = parallelize_on_rows(books_df, total_shelves_count)
books_df['is_series'] = parallelize_on_rows(books_df, is_series)

In [22]:
books_df.head(2)

,text_reviews_count,series,popular_shelves,is_ebook,average_rating,similar_books,description,format,link,authors,publisher,num_pages,publication_year,url,image_url,book_id,ratings_count,title,title_without_series,total_shelves_count,is_series
0,7.0,['189911'],"[{'count': '58', 'name': 'to-read'}, {'count':...",False,4.03,"['19997', '828466', '1569323', '425389', '1176...",Omnibus book club edition containing the Ladie...,Hardcover,https://www.goodreads.com/book/show/7327624-th...,"[{'author_id': '10333', 'role': ''}]","Nelson Doubleday, Inc.",600.0,1987.0,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140.0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ...",157,Y
1,3282.0,[],"[{'count': '7615', 'name': 'to-read'}, {'count...",False,3.49,"['6604176', '6054190', '2285777', '82641', '75...",Addie Downs and Valerie Adler were eight when ...,Hardcover,https://www.goodreads.com/book/show/6066819-be...,"[{'author_id': '9212', 'role': ''}]",Atria Books,368.0,2009.0,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...,6066819,51184.0,Best Friends Forever,Best Friends Forever,11416,N


## Merge reviews with books

In [23]:
reviews_10k_df = pd.read_csv("../data/reviews_cleaned_ENG_10k.csv")

reviews_10k_df.head(2)

,book_id,review_text,rating,n_votes,read_duration
0,2,"[""Rowling really hits it out of the park with ...",5.0,0,1512.0000
1,3,"[""I remember trying 3 times to read this but I...",4.0,0,43.7325


In [24]:
base_10k_df = reviews_10k_df.merge(books_df, on="book_id", how="left")
base_10k_df.head(2)

,book_id,review_text,rating,n_votes,read_duration,text_reviews_count,series,popular_shelves,is_ebook,average_rating,similar_books,description,format,link,authors,publisher,num_pages,publication_year,url,image_url,ratings_count,title,title_without_series,total_shelves_count,is_series
0,2,"[""Rowling really hits it out of the park with ...",5.0,0,1512.0000,23679.0,['163765'],"[{'count': '5299', 'name': 'currently-reading'...",False,4.47,"['13925', '319644', '13833', '40158', '116561'...",Harry Potter is due to start his fifth year at...,Paperback,https://www.goodreads.com/book/show/2.Harry_Po...,"[{'author_id': '1077326', 'role': ''}, {'autho...",Scholastic Inc.,870.0,2004.0,https://www.goodreads.com/book/show/2.Harry_Po...,https://images.gr-assets.com/books/1507396732m...,1766895.0,Harry Potter and the Order of the Phoenix (Har...,Harry Potter and the Order of the Phoenix (Har...,56224,Y
1,3,"[""I remember trying 3 times to read this but I...",4.0,0,43.7325,59856.0,['167817'],"[{'count': '525817', 'name': 'to-read'}, {'cou...",False,4.45,"['13830', '127586', '121822', '37586', '616435...",Harry Potter's life is miserable. His parents ...,Hardcover,https://www.goodreads.com/book/show/3.Harry_Po...,"[{'author_id': '1077326', 'role': ''}, {'autho...",Scholastic Inc,320.0,1997.0,https://www.goodreads.com/book/show/3.Harry_Po...,https://images.gr-assets.com/books/1474154022m...,4765497.0,Harry Potter and the Sorcerer's Stone (Harry P...,Harry Potter and the Sorcerer's Stone (Harry P...,771828,Y


## Base Dataset

In [72]:
base_1M_df = pd.read_csv("../data/base_ENG_1M.csv")
review_text = base_1M_df.head(1)["review_text"].values[0]

In [73]:
base_1M_df["n_reviews"] = base_1M_df["review_text"].apply(lambda x: len(ast.literal_eval(x)))
base_1M_df.sort_values("n_reviews", ascending=False).head(2)

,book_id,review_text,n_votes,read_duration,series,average_rating,similar_books,description,publisher,num_pages,publication_year,url,image_url,ratings_count,title,total_shelves_count,is_series,author_names,n_reviews
66252,11870085,"[""So.. well I know everyone's obsessed with th...",731,269.235026,[],4.26,"['10051706', '11418182', '10327303', '9627755'...","There is an alternate cover edition .\n""I fel...",Dutton Books,313.0,2012.0,https://www.goodreads.com/book/show/11870085-t...,https://images.gr-assets.com/books/1360206420m...,2429317.0,The Fault in Our Stars,197552,N,['John Green'],813
36702,2767052,"[""I cracked and finally picked this up. Very e...",384,173.646113,['339872'],4.34,"['1902241', '146499', '954674', '9917938', '10...",Winning will make you famous.\nLosing means ce...,Scholastic Press,374.0,2008.0,https://www.goodreads.com/book/show/2767052-th...,https://images.gr-assets.com/books/1447303603m...,4899965.0,"The Hunger Games (The Hunger Games, #1)",313443,Y,['Suzanne Collins'],602


In [77]:
potter_reviews =base_1M_df[base_1M_df["book_id"] == 2]["review_text"].values

In [80]:
potter_reviews[0]

'["Rowling really hits it out of the park with the characters and story in this volume. It is not bogged down by scenes involving Voldemort. No, the story\'s drama is driven by a human character (Professor Umbridge) who is easily recognized and hated as she continually ups the ante in a quiet battle of wills against Gryffindor and the good professors of Hogwarts. After the initial action and excitement, the story drags a bit with Sirius Black\'s family history, but after the students arrive in Hogwarts, the story never lets up its pace. First rate! Oh--and my boys loved it as well!", \'This is my least favorite of the series. Harry is written like a typical 15 year old - angsty, whiny, and melodramatic! It makes it a hard story to get through because you just want Harry to suck it up and get over the little stuff so that he can deal with the bigger issues at hand!\', "First of all, Im in love with these books since I was 12 years old, and I got my first book for Christmas. I was hooked

In [82]:
ast.literal_eval(potter_reviews[0])

["Rowling really hits it out of the park with the characters and story in this volume. It is not bogged down by scenes involving Voldemort. No, the story's drama is driven by a human character (Professor Umbridge) who is easily recognized and hated as she continually ups the ante in a quiet battle of wills against Gryffindor and the good professors of Hogwarts. After the initial action and excitement, the story drags a bit with Sirius Black's family history, but after the students arrive in Hogwarts, the story never lets up its pace. First rate! Oh--and my boys loved it as well!",
 'This is my least favorite of the series. Harry is written like a typical 15 year old - angsty, whiny, and melodramatic! It makes it a hard story to get through because you just want Harry to suck it up and get over the little stuff so that he can deal with the bigger issues at hand!',
 "First of all, Im in love with these books since I was 12 years old, and I got my first book for Christmas. I was hooked th

In [62]:
type(review_text)

str

In [53]:
description_text = base_10k_df.head(1)["description"].values[0]

In [54]:
description_text

'Harry Potter is due to start his fifth year at Hogwarts School of Witchcraft and Wizardry. His best friends Ron and Hermione have been very secretive all summer and he is desperate to get back to school and find out what has been going on. However, what Harry discovers is far more devastating than he could ever have expected...\nSuspense, secrets and thrilling action from the pen of J.K. Rowling ensure an electrifying adventure that is impossible to put down.'

In [45]:
base_10k_df = pd.read_csv("../data/base_ENG_all.csv")
base_10k_df.head(2)

,book_id,review_text,n_votes,read_duration,series,average_rating,similar_books,description,publisher,num_pages,publication_year,url,image_url,ratings_count,title,total_shelves_count,is_series,author_names
0,1,"[""First of all, Im in love with these books si...",1288,348.994088,['624922'],4.54,"['153795', '201341', '227865', '84119', '86737...",The war against Voldemort is not going well: e...,Scholastic Inc.,652.0,2006.0,https://www.goodreads.com/book/show/1.Harry_Po...,https://images.gr-assets.com/books/1361039191m...,1713866.0,Harry Potter and the Half-Blood Prince (Harry ...,370090,Y,"['J.K. Rowling', 'Mary GrandPre']"
1,2,"[""Rowling really hits it out of the park with ...",2233,621.952282,['163765'],4.47,"['13925', '319644', '13833', '40158', '116561'...",Harry Potter is due to start his fifth year at...,Scholastic Inc.,870.0,2004.0,https://www.goodreads.com/book/show/2.Harry_Po...,https://images.gr-assets.com/books/1507396732m...,1766895.0,Harry Potter and the Order of the Phoenix (Har...,56224,Y,"['J.K. Rowling', 'Mary GrandPre']"


## Upload to big query

In [27]:
from google.cloud import bigquery
import pandas as pd

PROJECT = "my-shelves-493916"
DATASET = "books_dataset"
TABLE = "base_reviews_ENG_10k"

table = f"{PROJECT}.{DATASET}.{TABLE}"


client = bigquery.Client()

write_mode = "WRITE_TRUNCATE" # or "WRITE_APPEND"
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(base_10k_df, table, job_config=job_config)
result = job.result()

/home/chris/.pyenv/versions/my_shelves_env/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [28]:
print(result)

LoadJob<project=my-shelves-493916, location=europe-west1, id=4741e9ea-c219-4963-9ccb-7883a0d0c618>


In [ ]:
cols_to_remove = ["rating",
                  "text_reviews_count",
                  "popular_shelves",
                  "is_ebook",
                  "format",
                  "link",
                  "title_without_series"]

## Authors

In [4]:
authors_df = pd.read_json("../data/kaggle/goodreads_book_authors.json", lines=True)

In [9]:
authors_df.head(2)

,average_rating,author_id,text_reviews_count,name,ratings_count
0,3.98,604031,7,Ronald J. Fields,49
1,4.08,626222,28716,Anita Diamant,546796


In [6]:
base_10k_df = pd.read_csv("../data/base_ENG_10k.csv")
base_10k_df.head(2)

,book_id,review_text,rating,n_votes,read_duration,text_reviews_count,series,popular_shelves,is_ebook,average_rating,similar_books,description,format,link,authors,publisher,num_pages,publication_year,url,image_url,ratings_count,title,title_without_series,total_shelves_count,is_series
0,2,"[""Rowling really hits it out of the park with ...",5.0,0,1512.0000,23679.0,['163765'],"[{'count': '5299', 'name': 'currently-reading'...",False,4.47,"['13925', '319644', '13833', '40158', '116561'...",Harry Potter is due to start his fifth year at...,Paperback,https://www.goodreads.com/book/show/2.Harry_Po...,"[{'author_id': '1077326', 'role': ''}, {'autho...",Scholastic Inc.,870.0,2004.0,https://www.goodreads.com/book/show/2.Harry_Po...,https://images.gr-assets.com/books/1507396732m...,1766895.0,Harry Potter and the Order of the Phoenix (Har...,Harry Potter and the Order of the Phoenix (Har...,56224,Y
1,3,"[""I remember trying 3 times to read this but I...",4.0,0,43.7325,59856.0,['167817'],"[{'count': '525817', 'name': 'to-read'}, {'cou...",False,4.45,"['13830', '127586', '121822', '37586', '616435...",Harry Potter's life is miserable. His parents ...,Hardcover,https://www.goodreads.com/book/show/3.Harry_Po...,"[{'author_id': '1077326', 'role': ''}, {'autho...",Scholastic Inc,320.0,1997.0,https://www.goodreads.com/book/show/3.Harry_Po...,https://images.gr-assets.com/books/1474154022m...,4765497.0,Harry Potter and the Sorcerer's Stone (Harry P...,Harry Potter and the Sorcerer's Stone (Harry P...,771828,Y


In [11]:
authors_df[["author_id", "name"]].head(2)

,author_id,name
0,604031,Ronald J. Fields
1,626222,Anita Diamant


In [26]:
def get_name_by_author_id(author_id):
    return authors_df[authors_df["author_id"] == author_id]["name"].values[0]
get_name_by_author_id(604031)

'Ronald J. Fields'

In [40]:
def get_author_names(row):
    authors = ast.literal_eval(row['authors'])
    author_names = []
    for author in authors:
        author_id = int(author['author_id'])
        author_name = get_name_by_author_id(author_id)
        author_names.append(author_name)
    return author_names

In [13]:
import ast

In [32]:
l = base_10k_df.iloc[0]["authors"]

In [41]:
get_author_names(base_10k_df.iloc[0])

['J.K. Rowling', 'Mary GrandPre']

In [38]:
get_name_by_author_id(int(ast.literal_eval(base_10k_df.iloc[0]["authors"])[0]['author_id']))

'J.K. Rowling'

In [43]:
base_10k_df["author_names"] = base_10k_df.apply(get_author_names, axis=1)
base_10k_df.sample(7)

,book_id,review_text,rating,n_votes,read_duration,text_reviews_count,series,popular_shelves,is_ebook,average_rating,similar_books,description,format,link,authors,publisher,num_pages,publication_year,url,image_url,ratings_count,title,title_without_series,total_shelves_count,is_series,author_id,author_names
3971,22980192,"[""I was really excited to read this book. Devi...",2.0,0,48.000000,51.0,['809932'],"[{'count': '466', 'name': 'to-read'}, {'count'...",False,3.44,"['23582746', '24224363', '25224283', '27234195...","Lucinda is as old as humanity itself, yet perp...",Paperback,https://www.goodreads.com/book/show/22980192-d...,"[{'author_id': '13499400', 'role': ''}, {'auth...",Ghost Mountain Books,300.0,2015.0,https://www.goodreads.com/book/show/22980192-d...,https://images.gr-assets.com/books/1420789187m...,79.0,"Devil's Daughter (Lucinda's Pawnshop, #1)","Devil's Daughter (Lucinda's Pawnshop, #1)",596,Y,"{'author_id': '13499400', 'role': ''}","[Hope Schenk-de Michele, Paul Marquez, Maya Ka..."
1600,7378426,['If you read through the reviews for this boo...,4.0,0,205.544722,132.0,['245831'],"[{'count': '701', 'name': 'to-read'}, {'count'...",False,3.35,"['6892166', '6900601', '5037673', '12987245', ...",Lenk can barely keep control of his mismatched...,Hardcover,https://www.goodreads.com/book/show/7378426-to...,"[{'author_id': '3252210', 'role': ''}]",Victor Gollancz Science Fiction,612.0,2010.0,https://www.goodreads.com/book/show/7378426-to...,https://images.gr-assets.com/books/1327895166m...,1102.0,"Tome of the Undergates (Aeons' Gate, #1)","Tome of the Undergates (Aeons' Gate, #1)",1317,Y,"{'author_id': '3252210', 'role': ''}",[Sam Sykes]
964,776159,"['This was my first Chandler, and I became an ...",5.0,2,216.000000,563.0,['855279'],"[{'count': '416', 'name': 'mystery'}, {'count'...",False,4.06,"['30004', '648640', '45338']",A couple of missing wives--one a rich man's an...,Paperback,https://www.goodreads.com/book/show/776159.The...,"[{'author_id': '1377', 'role': ''}]",Vintage Crime/Black Lizard,266.0,1988.0,https://www.goodreads.com/book/show/776159.The...,https://images.gr-assets.com/books/1501530591m...,11889.0,"The Lady in the Lake (Philip Marlowe, #4)","The Lady in the Lake (Philip Marlowe, #4)",2486,Y,"{'author_id': '1377', 'role': ''}",[Raymond Chandler]
3709,21411058,"[""5 perfect stars!!! \n Well. Where do I begin...",5.0,9,72.000000,2300.0,['673726'],"[{'count': '2192', 'name': 'to-read'}, {'count...",True,4.09,"['22888864', '26869616', '21170685', '17793815...",When Cassie Taylor met Ethan Holt at acting sc...,Kindle Edition,https://www.goodreads.com/book/show/21411058-b...,"[{'author_id': '8034143', 'role': ''}]",St. Martin's Griffin,417.0,2014.0,https://www.goodreads.com/book/show/21411058-b...,https://images.gr-assets.com/books/1413547825m...,14692.0,"Bad Romeo (Starcrossed, #1)","Bad Romeo (Starcrossed, #1)",5706,Y,"{'author_id': '8034143', 'role': ''}",[Leisa Rayven]
4311,25439738,"[""I absolutely fell in love with this book and...",5.0,17,24.000000,93.0,[],"[{'count': '2586', 'name': 'to-read'}, {'count...",False,3.75,"['28270232', '26841614', '26248223', '30194656...",#1 BEST SELLING AMAZON KINDLE BOOK in DIARIES ...,Paperback,https://www.goodreads.com/book/show/25439738-a...,"[{'author_id': '13856997', 'role': ''}]",English House,333.0,2015.0,https://www.goodreads.com/book/show/25439738-a...,https://images.gr-assets.com/books/1430167528m...,255.0,Affairytale,Affairytale,2880,N,"{'author_id': '13856997', 'role': ''}",[C.J. English]
125,7244,['** spoiler alert ** \n I picked this one up ...,5.0,0,0.000000,18075.0,[],"[{'count': '32943', 'name': 'to-read'}, {'coun...",False,4.03,"['7742', '5174', '292408', '5205', '158674', '...",The Poisonwood Bible is a story told by the wi...,Hardcover,https://www.goodreads.com/book/show/7244.The_P...,"[{'author_id': '3541', 'role': ''}]",Harper Perennial Modern Classics,546.0,2005.0,https://www.goodreads.com/book/show/7244.The_P...,https://images.gr-assets.com/books/141

In [ ]:
base_10k_df.merge(authors_df[["author_id", "name"]], )

In [5]:
authors_df.head(2)

,average_rating,author_id,text_reviews_count,name,ratings_count
0,3.98,604031,7,Ronald J. Fields,49
1,4.08,626222,28716,Anita Diamant,546796


## Explore Datasets

In [166]:
from my_shelves.utils.dataset import get_books, get_reviews

In [167]:
books_df = get_books(data_dir="../data")
reviews_df = get_reviews(data_dir="../data")

In [168]:
books_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,NaN,7,['189911'],US,eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",B00071IKUY,False,4.03,NaN,...,NaN,Book Club Edition,1987.0,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140,8948723,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,0743294297,3282,[],US,eng,"[{'count': '7615', 'name': 'to-read'}, {'count...",NaN,False,3.49,B002ENBLOK,...,7.0,NaN,2009.0,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...,6066819,51184,6243154,Best Friends Forever,Best Friends Forever
2,NaN,60,['1052227'],US,eng,"[{'count': '54', 'name': 'currently-reading'},...",B01NCIKAQX,True,4.33,B01NCIKAQX,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,33394837,269,54143148,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2)
3,555118000X,19,[],US,eng,"[{'count': '3488', 'name': 'to-read'}, {'count...",NaN,False,3.82,NaN,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/89373.The_...,https://s.gr-assets.com/assets/nophoto/book/11...,89373,77,1080201,The Bonfire of the Vanities,The Bonfire of the Vanities
4,0842379428,566,[],US,eng,"[{'count': '6393', 'name': 'to-read'}, {'count...",NaN,False,4.26,B000FCKCJC,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/89376.Heaven,https://images.gr-assets.com/books/1406508230m...,89376,7345,86257,Heaven,Heaven


In [169]:
reviews_df.head()

,user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments
0,8842281e1d1347389f2ab93d60773d4d,19398490,ea4a220b10e6b5c796dae0e3b970aff1,4,A beautiful story. It is rare to encounter a b...,Sun Jan 03 21:20:46 -0800 2016,Tue Sep 20 23:30:15 -0700 2016,Tue Sep 13 11:51:51 -0700 2016,Sat Aug 20 07:03:03 -0700 2016,35,5
1,8842281e1d1347389f2ab93d60773d4d,8664353,da2d4cfce836a2c57ad55c38437aa692,5,"Wow. Amazing story, and well told - kept me up...",Sun Nov 20 09:10:15 -0800 2011,Wed Mar 22 11:47:04 -0700 2017,Wed May 16 00:00:00 -0700 2012,Sun May 06 00:00:00 -0700 2012,50,4
2,01ec1a320ffded6b2dd47833f2c8e4fb,27878417,b22dd337ee1938dd7cac80261aed6d2f,5,Contains major spoilers so only read this afte...,Mon Nov 16 21:35:17 -0800 2015,Sun Dec 06 22:45:19 -0800 2015,Wed Nov 18 11:06:15 -0800 2015,Mon Nov 16 00:00:00 -0800 2015,16,0
3,01ec1a320ffded6b2dd47833f2c8e4fb,23518092,39afe8c459a61c9d66fe212217608d72,4,"This is a juicy, fast-paced and totally enjoya...",Sun Jan 18 22:08:52 -0800 2015,Sun Jan 18 22:09:49 -0800 2015,Sun Jan 18 00:00:00 -0800 2015,NaN,6,2
4,01ec1a320ffded6b2dd47833f2c8e4fb,23350137,544dca1efd8819f93996a20fec44f199,4,"This is a fun, loving, sexy and very satisfyin...",Sun Nov 30 23:05:08 -0800 2014,Tue Dec 02 08:25:58 -0800 2014,Mon Dec 01 00:00:00 -0800 2014,Sun Nov 30 00:00:00 -0800 2014,6,0


In [170]:
reviews_df.shape

(110000, 11)

In [171]:
def clean_reviews_text(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    df = df.dropna(subset=[column_name])
    df = df[df[column_name].str.strip() != ""]
    return df

In [172]:
reviews_df = clean_reviews_text(reviews_df, column_name="review_text")
reviews_df.shape

(109963, 11)

In [173]:
def add_reviews_count_words(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    df[f"{column_name}_n_words"] = df[column_name].apply(lambda x: len(x.split()))
    return df

In [174]:
reviews_df = add_reviews_count_words(reviews_df, column_name="review_text")
reviews_df.shape

(109963, 12)

In [175]:
def clean_small_reviews(df: pd.DataFrame, column_name: str, min_words: int) -> pd.DataFrame:
    df = df[df[f"{column_name}_n_words"] > min_words]
    return df

In [176]:
reviews_df = clean_small_reviews(reviews_df, column_name="review_text", min_words=50)

Remove useless columns

In [177]:
def convert_column_to_datetime(df: pd.DataFrame, column_name: str, replacement_value: str="futur") -> pd.DataFrame:
    replacement_date = 'Sun Jul 1 00:00:00 -0700 1970'
    if replacement_value == "futur":
        replacement_date = 'Sun Jul 1 00:00:00 -0700 2050'
    df[column_name] = pd.to_datetime(df[column_name],
                                     format='%a %b %d %H:%M:%S %z %Y',
                                     utc=True,
                                     errors='coerce')
    df[column_name] = df[column_name].fillna(value=replacement_date)
    return df

In [178]:
def clean_reviews_dates(df: pd.DataFrame) -> pd.DataFrame:
    date_columns = {"started_at": "past",
                    "read_at": "futur",
                    "date_added": "futur",
                    "date_updated": "futur"}
    for column, replacement_value in date_columns.items():
        df = convert_column_to_datetime(df,
                                        column_name=column,
                                        replacement_value=replacement_value)
    return df

In [179]:
reviews_df = clean_reviews_dates(reviews_df)

In [180]:
def compute_read_duration(df: pd.DataFrame, start_column: str, end_column: str) -> pd.DataFrame:
    df["read_duration"] = (df[end_column] - df[start_column]).dt.total_seconds() / 3600
    return df

In [181]:
reviews_df = compute_read_duration(reviews_df, start_column="started_at", end_column="read_at")
reviews_df.head()

,user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments,review_text_n_words,read_duration
0,8842281e1d1347389f2ab93d60773d4d,19398490,ea4a220b10e6b5c796dae0e3b970aff1,4,A beautiful story. It is rare to encounter a b...,2016-01-04 05:20:46+00:00,2016-09-21 06:30:15+00:00,2016-09-13 18:51:51+00:00,2016-08-20 14:03:03+00:00,35,5,258,580.813333
1,8842281e1d1347389f2ab93d60773d4d,8664353,da2d4cfce836a2c57ad55c38437aa692,5,"Wow. Amazing story, and well told - kept me up...",2011-11-20 17:10:15+00:00,2017-03-22 18:47:04+00:00,2012-05-16 07:00:00+00:00,2012-05-06 07:00:00+00:00,50,4,264,240.000000
3,01ec1a320ffded6b2dd47833f2c8e4fb,23518092,39afe8c459a61c9d66fe212217608d72,4,"This is a juicy, fast-paced and totally enjoya...",2015-01-19 06:08:52+00:00,2015-01-19 06:09:49+00:00,2015-01-18 08:00:00+00:00,1970-07-01 07:00:00+00:00,6,2,144,390529.000000
4,01ec1a320ffded6b2dd47833f2c8e4fb,23350137,544dca1efd8819f93996a20fec44f199,4,"This is a fun, loving, sexy and very satisfyin...",2014-12-01 07:05:08+00:00,2014-12-02 16:25:58+00:00,2014-12-01 08:00:00+00:00,2014-11-30 08:00:00+00:00,6,0,216,24.000000
5,01ec1a320ffded6b2dd47833f2c8e4fb,22911904,1427adba9fafea91142088973a49927b,4,I loved this book!! Pure enjoyment!! Love the ...,2014-08-13 20:42:16+00:00,2014-11-10 16:15:42+00:00,2014-11-09 08:00:00+00:00,2014-11-09 08:00:00+00:00,3,3,305,0.000000


In [36]:
reviews_df.shape

(62289, 13)

In [ ]:
cols_to_drop = ["user_id", "date_updated"]

In [1]:
import os
os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

'/home/chris/code/christophezito/gcp/my-shelves-493916-4fc17c3c5ea6.json'

In [183]:
from my_shelves.utils.params import GCP_PROJECT, BQ_DATASET
from my_shelves.utils.bigquery import get_book

df = get_book(22077083)
df


/home/chris/.pyenv/versions/my_shelves_env/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,None,1,['161354'],US,eng,"[{'count': '559', 'name': 'to-read'}, {'count'...",None,False,3.8,None,...,NaN,None,1992.0,https://www.goodreads.com/book/show/22077083-t...,https://images.gr-assets.com/books/1399953592m...,22077083,5,2232945,The Haunted Fort (The Hardy Boys # 03),The Haunted Fort (The Hardy Boys # 03)


In [17]:
import pandas as pd

In [25]:
reviews_clean_df = pd.read_csv("../data/reviews_cleaned.csv")
reviews_clean_df.head()

,book_id,review_text,rating,n_votes,read_duration
0,1362,['Did you know that Egyptian women urinate sta...,4.103448,35,366.249693
1,1365,['It may be surprising to find out that one of...,4.000000,2,500.532870
2,1366,"[""I'm a historian and this is the book that st...",4.400000,10,4018.185778
3,1367,"[""Refreshingly amusing. In the histories, Hero...",4.000000,1,2049.278056
4,1368,"['Herodotus was hailed as ""The Father of Histo...",5.000000,6,4992.000000


In [26]:
reviews_clean_df.shape

(6879, 5)

In [22]:
df_grouped = reviews_clean_df.groupby("book_id").agg({
    "review_text": list,
    "rating": "mean",
    "n_votes": "sum",
    "read_duration": "mean"
}).reset_index()

In [23]:
df_grouped.head()

,book_id,review_text,rating,n_votes,read_duration
0,1362,[Did you know that Egyptian women urinate stan...,3.0,3,0.0
1,5345,[Ho-hum. Not that thrilling. Non-fiction book ...,2.0,0,0.0
2,5346,[I loved John Grisham when he first hit the li...,4.0,0,216.0
3,13872,"[I had to look up ""geek"" because it surely did...",4.0,4,24.0
4,14069,[Sent from China on a journey to Istanbul to r...,4.0,0,0.0


In [2]:
import pandas as pd

In [5]:
pd.read_csv("../data/reviews_ENG_mini.csv").shape

(110000, 11)

In [7]:
pd.read_csv("../data/goodreads_reviews_ENG_10k.csv").shape

(10000, 11)

In [8]:
pd.read_csv("../data/goodreads_reviews_ENG_100k.csv").shape

(100000, 11)

In [13]:
pd.read_csv("../data/goodreads_reviews_ENG_1M.csv").shape

(1000000, 11)

In [4]:
pd.read_csv("../data/goodreads_reviews_ENG_all.csv").shape

(11409758, 11)

## Cleaned Reviews

In [16]:
pd.read_csv("../data/reviews_cleaned_ENG_10k.csv").shape

(5019, 5)

In [17]:
pd.read_csv("../data/reviews_cleaned_ENG_100k.csv").shape

(37774, 5)

In [18]:
pd.read_csv("../data/reviews_cleaned_ENG_1M.csv").shape

(189563, 5)

In [19]:
reviews_clean_1m_df = pd.read_csv("../data/reviews_cleaned_ENG_1M.csv")
reviews_clean_1m_df.head()

,book_id,review_text,rating,n_votes,read_duration
0,1,"[""First of all, Im in love with these books si...",4.487500,34,176.873063
1,2,"[""Rowling really hits it out of the park with ...",4.388350,243,793.423830
2,3,"[""I remember trying 3 times to read this but I...",4.423729,259,435.510829
3,4,"[""I actually read this book first, thinking it...",5.000000,4,258.948750
4,5,"[""Rowling does it again with Harry and the gan...",4.636364,79,744.728725


In [5]:
reviews_clean_all_df = pd.read_csv("../data/reviews_cleaned_ENG_all.csv")
reviews_clean_all_df.head()

,book_id,review_text,rating,n_votes,read_duration
0,1,"[""First of all, Im in love with these books si...",4.487402,1288,348.994088
1,2,"[""Rowling really hits it out of the park with ...",4.328467,2233,621.952282
2,3,"[""I remember trying 3 times to read this but I...",4.437788,4964,414.159174
3,4,"[""I actually read this book first, thinking it...",4.529412,80,156.405964
4,5,"[""Rowling does it again with Harry and the gan...",4.582845,2022,320.308168


In [6]:
reviews_clean_all_df.shape

(655288, 5)

## Books dataset

In [16]:
books_df_10k = pd.read_csv("../data/raw/goodreads_books_ENG.csv")
books_df_10k.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,NaN,7.0,['189911'],US,eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",B00071IKUY,False,4.03,NaN,...,NaN,Book Club Edition,1987.0,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140.0,8948723.0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,0743294297,3282.0,[],US,eng,"[{'count': '7615', 'name': 'to-read'}, {'count...",NaN,False,3.49,B002ENBLOK,...,7.0,NaN,2009.0,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...,6066819,51184.0,6243154.0,Best Friends Forever,Best Friends Forever
2,NaN,60.0,['1052227'],US,eng,"[{'count': '54', 'name': 'currently-reading'},...",B01NCIKAQX,True,4.33,B01NCIKAQX,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,33394837,269.0,54143148.0,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2)
3,555118000X,19.0,[],US,eng,"[{'count': '3488', 'name': 'to-read'}, {'count...",NaN,False,3.82,NaN,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/89373.The_...,https://s.gr-assets.com/assets/nophoto/book/11...,89373,77.0,1080201.0,The Bonfire of the Vanities,The Bonfire of the Vanities
4,0842379428,566.0,[],US,eng,"[{'count': '6393', 'name': 'to-read'}, {'count...",NaN,False,4.26,B000FCKCJC,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/89376.Heaven,https://images.gr-assets.com/books/1406508230m...,89376,7345.0,86257.0,Heaven,Heaven


In [ ]:
features_to_keep = ['book_id','description','publication_year','image_url','url','title','num_pages','series', 'ratings_count']

## Extended Reviews

In [25]:
reviews_clean_10k_df = pd.read_csv("../data/reviews_cleaned_ENG_10k.csv")
reviews_clean_10k_df.head()

,book_id,review_text,rating,n_votes,read_duration
0,2,"[""Rowling really hits it out of the park with ...",5.0,0,1512.0000
1,3,"[""I remember trying 3 times to read this but I...",4.0,0,43.7325
2,11,"['Good book for when you are sick in bed, or o...",3.0,0,0.0000
3,21,"[""A fascinating history of science. Ever curio...",3.0,46,0.0000
4,27,['Bryson is my kind of travel writer.He rarely...,4.5,0,84.0000


In [26]:
reviews_extend_10k_df = pd.read_csv("../data/reviews_extended_ENG_10k.csv")
reviews_extend_10k_df.head()

,book_id,review_text,rating,n_votes,read_duration,city,type,country,continent,province
0,2,"[""Rowling really hits it out of the park with ...",5.0,0,1512.0000,b'North Judithbury',b'mountain',b'Peru',b'Americas',b'Amazonas'
1,3,"[""I remember trying 3 times to read this but I...",4.0,0,43.7325,b'East Jill',b'forest',b'Saint Vincent and the Grenadines',b'Americas',b'Grenadines'
2,11,"['Good book for when you are sick in bed, or o...",3.0,0,0.0000,b'New Roberttown',b'beach',b'Cayman Islands',b'Americas',b'Eastern'
3,21,"[""A fascinating history of science. Ever curio...",3.0,46,0.0000,b'East Jessetown',b'mountain',b'Saint Kitts and Nevis',b'Americas',b'Saint Paul Charlestown'
4,27,['Bryson is my kind of travel writer.He rarely...,4.5,0,84.0000,b'Lake Debra',b'mountain',b'Guadeloupe',b'Americas',b'Basse-Terre'


In [28]:
reviews_clean_10k_df.shape

(5019, 5)

In [27]:
reviews_extend_10k_df.shape

(5019, 10)

In [7]:
reviews_extend_100k_df = pd.read_csv("../data/reviews_extended_ENG_100k.csv")
reviews_extend_100k_df.head()

,book_id,review_text,rating,n_votes,read_duration,city,type,country,continent,province
0,1,"[""First of all, Im in love with these books si...",4.714286,13,69.431071,b'North Judithbury',b'mountain',b'Peru',b'Americas',b'Amazonas'
1,2,"[""Rowling really hits it out of the park with ...",4.470588,31,393.387778,b'East Jill',b'forest',b'Saint Vincent and the Grenadines',b'Americas',b'Grenadines'
2,3,"[""I remember trying 3 times to read this but I...",4.210526,32,75.433523,b'New Roberttown',b'beach',b'Cayman Islands',b'Americas',b'Eastern'
3,4,"[""I actually read this book first, thinking it...",5.000000,0,0.000000,b'East Jessetown',b'mountain',b'Saint Kitts and Nevis',b'Americas',b'Saint Paul Charlestown'
4,5,"[""Rowling does it again with Harry and the gan...",4.666667,4,219.603542,b'Lake Debra',b'mountain',b'Guadeloupe',b'Americas',b'Basse-Terre'


In [10]:
reviews_extend_100k_df.shape

(37774, 10)

In [11]:
reviews_extend_1M_df = pd.read_csv("../data/reviews_extended_ENG_1M.csv")
reviews_extend_1M_df.head()

,book_id,review_text,rating,n_votes,read_duration,city,type,country,continent,province
0,1,"[""First of all, Im in love with these books si...",4.487500,34,176.873063,b'North Judithbury',b'mountain',b'Peru',b'Americas',b'Amazonas'
1,2,"[""Rowling really hits it out of the park with ...",4.388350,243,793.423830,b'East Jill',b'forest',b'Saint Vincent and the Grenadines',b'Americas',b'Grenadines'
2,3,"[""I remember trying 3 times to read this but I...",4.423729,259,435.510829,b'New Roberttown',b'beach',b'Cayman Islands',b'Americas',b'Eastern'
3,4,"[""I actually read this book first, thinking it...",5.000000,4,258.948750,b'East Jessetown',b'mountain',b'Saint Kitts and Nevis',b'Americas',b'Saint Paul Charlestown'
4,5,"[""Rowling does it again with Harry and the gan...",4.636364,79,744.728725,b'Lake Debra',b'mountain',b'Guadeloupe',b'Americas',b'Basse-Terre'


In [12]:
reviews_extend_1M_df.shape

(189563, 10)

In [13]:
reviews_extend_all_df = pd.read_csv("../data/reviews_extended_ENG_all.csv")
reviews_extend_all_df.head()

,book_id,review_text,rating,n_votes,read_duration,city,type,country,continent,province
0,1,"[""First of all, Im in love with these books si...",4.487402,1288,348.994088,b'North Judithbury',b'mountain',b'Peru',b'Americas',b'Amazonas'
1,2,"[""Rowling really hits it out of the park with ...",4.328467,2233,621.952282,b'East Jill',b'forest',b'Saint Vincent and the Grenadines',b'Americas',b'Grenadines'
2,3,"[""I remember trying 3 times to read this but I...",4.437788,4964,414.159174,b'New Roberttown',b'beach',b'Cayman Islands',b'Americas',b'Eastern'
3,4,"[""I actually read this book first, thinking it...",4.529412,80,156.405964,b'East Jessetown',b'mountain',b'Saint Kitts and Nevis',b'Americas',b'Saint Paul Charlestown'
4,5,"[""Rowling does it again with Harry and the gan...",4.582845,2022,320.308168,b'Lake Debra',b'mountain',b'Guadeloupe',b'Americas',b'Basse-Terre'


In [14]:
reviews_extend_all_df.shape

(655288, 10)